### Session Results: Silver to Gold
Combine `formula1_incr.silver.results` and `formula1_incr.silver.sprints` into one unified fact table `formula1_incr.gold.facts_session_results`.

#### Setup
- `01.environment-config` → loads catalog name, silver/gold schema names
- `04.gold_helpers` → loads the `write_to_gold()` function we use to save data

In [0]:
%run ../00-common/01.environment-config 

In [0]:
dbutils.widgets.text('p_batch_id','')
v_batch_id= dbutils.widgets.get('p_batch_id')

In [0]:
%run ../00-common/04.gold_helpers 

#### Target Table

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as f

#### Target Table
- Set the full table name: `formula1_incr.gold.facts_session_results`

In [0]:
target_table = f'{catalog_name}.{gold_schema}.facts_session_results'

#### Read Silver Tables
- Read `results` and `sprints`, filter by batch, add `session_type` column, drop metadata columns

In [0]:
results_df = (spark.table(f'{catalog_name}.{silver_schema}.results')
              .filter(col('batch_id') == v_batch_id)
              .withColumn('session_type', lit('RACE'))
              .drop('race_name', 'race_date', 'ingestion_timestamp', 'source_file','batch_id','created_timestamp','updated_timestamp')
)

In [0]:
sprints_df = (spark.table(f'{catalog_name}.{silver_schema}.sprints')
              .filter(col('batch_id') == v_batch_id)
              .withColumn('session_type', lit('SPRINT'))
              .drop('race_name', 'race_date', 'ingestion_timestamp', 'source_file', 'batch_id','created_timestamp','updated_timestamp')
)

#### Union
- Stack both DataFrames into one using `unionByName()`

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

#### Derive Columns
- `is_win` → true if driver finished 1st
- `is_podium` → true if driver finished top 3
- `has_points` → true if driver scored points

In [0]:
facts_session_results_df =(results_sprints_df
 .withColumn('is_win',col('final_position') == 1)
 .withColumn('is_podium', col('final_position').between(1,3))
 .withColumn('has_points', col('points') > 0))


In [0]:
display(facts_session_results_df.filter('season == 2025'))

#### Write to Gold
- If the table doesn't exist, creates it from scratch
- If it exists, merges new/updated rows using `write_to_gold()`

In [0]:
write_to_gold(
    input_df= facts_session_results_df,
    target_table=target_table,
    merge_condition='''
                    t.season = s.season 
                     AND t.round = s.round 
                     AND t.driver_id = s.driver_id 
                     AND t.constructor_id = s.constructor_id
                     AND t.session_type = s.session_type''',

    columns_to_update=[
      "grid_position",
      "Completed_laps",
      "car_number",
      "points",
      "final_position",
      "final_position_text",
      "status",
      "is_win",
      "is_podium",
      "has_points"
    ]
)

In [0]:
display(spark.read.table(target_table))